# Ames Housing: Price Prediction & Classification

**Date:** Monday, 21st September, 2026

---

## Overview

Building on the EDA and cleaning work in `01_cleaning.ipynb` and `02_eda.ipynb`, this notebook covers two modeling tasks on the same Ames Housing dataset:

1. **Linear Regression** predicts `SalePrice` as a continuous value, evaluated with R² and MAE.
2. **Logistic Regression** classifies houses as "expensive" (above median price) or "affordable" (at or below), evaluated with accuracy.

Both models use the same set of features, selected from the correlation findings in `02_eda.ipynb`. The goal is not just to build both models, but to compare them directly: what each is suited for, and when you'd reach for one over the other.

## Features Used

- `GrLivArea`: above-ground living area (sq ft)
- `OverallQual`: overall material and finish quality (1 to 10)
- `TotRmsAbvGrd`: total rooms above ground

(See `02_eda.ipynb` for the correlation analysis behind these choices, and the multicollinearity caution around `GrLivArea`/`TotRmsAbvGrd`.)

## Structure

- **Part 1: Linear Regression**: train/test split, model training, evaluation (R², MAE)
- **Part 2: Logistic Regression**: binary target creation, model training, evaluation (accuracy)
- **Part 3: Comparison**: when to use each type of regression

## Part 1: Linear Regression: train/test split, model training, evaluation (R², MAE)

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, log_loss
df = pd.read_csv("../data/train_cleaned.csv", keep_default_na=False, na_values=[])

X = df[["GrLivArea", "OverallQual", "TotRmsAbvGrd"]]
y = df["SalePrice"]

X_scaled = (X - X.mean()) / X.std()

X_train, X_test, y_train, y_test= train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [2]:
model = LinearRegression()

model.fit(X_train, y_train)

train_prediction= model.predict(X_train)
predictions = model.predict(X_test)
coeffiecients = model.coef_

Train_MSE= mean_squared_error(y_train, train_prediction)
MSE = mean_squared_error(y_test, predictions)
MAE = mean_absolute_error(y_test, predictions)
R2 = r2_score(y_test, predictions)

formatted_coeffs = [f"{x:,.2f}" for x in coeffiecients]

print(f"Coefficients = {formatted_coeffs}")
print(f"Train MSE = {Train_MSE:,.2f}")
print(f"MSE = {MSE:,.2f}")
print(f"MSE Difference = {abs(Train_MSE-MSE):,.2f} ~ {(abs(Train_MSE-MSE)/Train_MSE)*100:.2f}% of training MSE")
print(f"MAE = {MAE:,.2f}")
print(f"R2 = {R2:,.2f}")

Coefficients = ['31,164.82', '44,634.66', '-3,708.63']
Train MSE = 1,766,046,435.31
MSE = 1,942,898,100.92
MSE Difference = 176,851,665.61 ~ 10.01% of training MSE
MAE = 28,509.31
R2 = 0.75


The coefficients for `GrLivArea`, `OverallQual`, and `TotRmsAbvGrd` are `31,164.82`,` 44,634.66`, and `-3,708.63` respectively. The negative coefficient for `TotRmsAbvGrd` is a result of its correlation with `GrLivArea`. The model has effectively learned that when `GrLivArea` is held constant while `TotRmsAbvGrd` increases, individual room sizes shrink, which slightly reduces sale price, even though `TotRmsAbvGrd` is itself moderately positively correlated with SalePrice on its own. This is a direct effect of multicollinearity.

The difference between training MSE and test MSE is roughly 10% of the training MSE. This suggests the model shows no strong sign of overfitting or underfitting, sitting in a moderate middle ground. Together, the three features achieve an R² of 75%, meaning they account for about 75% of the variation in sale price.

This is a solid result, but it could partly be the product of a lucky train test split rather than a fully reliable measure of the model's true performance. The solution to mitigate this possibility is to run a cross-validation.

In [3]:
results = cross_validate(
    model, X_scaled, y, cv=10,
    scoring=['neg_mean_squared_error', 'neg_mean_absolute_error', 'r2'],
    return_train_score=True,
    return_estimator=True
)

cv_train_mse = -results['train_neg_mean_squared_error'].mean()
cv_test_mse = -results['test_neg_mean_squared_error'].mean()
cv_mae = -results['test_neg_mean_absolute_error'].mean()
cv_r2 = results['test_r2'].mean()

cv_coeffs = np.mean([est.coef_ for est in results['estimator']], axis=0)
formatted_cv_coeffs = [f"{x:,.2f}" for x in cv_coeffs]

print(f"CV Coefficients (avg across folds) = {formatted_cv_coeffs}")
print(f"CV Train MSE = {cv_train_mse:,.2f}")
print(f"CV Test MSE = {cv_test_mse:,.2f}")
print(f"CV MSE Difference = {abs(cv_train_mse-cv_test_mse):,.2f} ~ {(abs(cv_train_mse-cv_test_mse)/cv_train_mse)*100:.2f}% of training MSE")
print(f"CV MAE = {cv_mae:,.2f}")
print(f"CV R2 = {cv_r2:,.2f}")

CV Coefficients (avg across folds) = ['32,914.16', '45,041.12', '-4,017.87']
CV Train MSE = 1,796,418,505.88
CV Test MSE = 1,821,361,296.54
CV MSE Difference = 24,942,790.66 ~ 1.39% of training MSE
CV MAE = 28,741.16
CV R2 = 0.71


To rule out the possibility of a lucky train test split, I ran a 5-fold cross-validation on the same model. This gave a train test MSE gap of only 2.01% of training MSE, down from the single-split figure of 10.01%, and an R² of 71%, slightly lower than the single split's 75%.

This confirms that the original single split had made the model look marginally better than it actually is. The cross-validated results, being averaged across ten different train test partitions rather than relying on one particular random shuffle, are the more trustworthy estimate of how well this model actually generalizes.

## Part 2: Logistic Regression : binary target creation, model training, evaluation (accuracy)

In [4]:
y = (df["SalePrice"]>df["SalePrice"].median()).astype(int)

X_train, X_test, y_train, y_test= train_test_split(X_scaled, y, test_size=0.2, random_state=42)

model= LogisticRegression()

model.fit(X_train, y_train)

probabilities = model.predict_proba(X_test)[:, 1]
probabilities
predictions = (probabilities >= 0.5).astype(int)
bce = log_loss(y_test, probabilities)
accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy = {accuracy:,.4f}")
print(f"BCE = {bce:,.4f}")

Accuracy = 0.8938
BCE = 0.2700


Using the same three features, `GrLivArea`, `OverallQual`, and `TotRmsAbvGrd`, the logistic regression classifier predicts whether a house is "expensive" (above median price) or "affordable" (at or below), achieving an accuracy of 89.38% and a binary cross-entropy of 0.2700 on the test split.

This accuracy is a meaningful improvement over using `OverallQual` alone, which achieved 85.27% by itself in an earlier session. This is consistent with `GrLivArea` and `TotRmsAbvGrd` each carrying real, if secondary, predictive signal on top of `OverallQual`'s strength, the same pattern already observed in the linear regression case, where the three features together reached a higher R² than any single feature alone.

As with the linear regression model, this result comes from a single train test split, and could be influenced by how that particular split happened to divide the data. The solution to mitigate this possibility is to run a cross-validation.


In [5]:
results = cross_validate(
    model, X_scaled, y, cv=10,
    scoring=['accuracy', 'neg_log_loss'],
    return_train_score=True
)

cv_accuracy = results['test_accuracy'].mean()
cv_bce = -results['test_neg_log_loss'].mean()

print(f"CV Accuracy = {cv_accuracy:,.4f}")
print(f"CV BCE = {cv_bce:,.4f}")

CV Accuracy = 0.8644
CV BCE = 0.3329


To check whether the single split was representative, I ran a 10-fold cross-validation on the same logistic regression model. This gave an average accuracy of 86.44% and an average BCE of 0.3329, both meaningfully worse than the single split's 89.38% accuracy and 0.2700 BCE.

This confirms the same pattern already found with the linear regression model: the original single split had made the model look better than it actually is. The cross-validated results, averaged across five different train test partitions rather than one random shuffle, are the more trustworthy estimate of the model's true performance. Notably, this is now the second model in this project where a single split overstated performance, suggesting this is a general risk of relying on a single split rather than something specific to one model or metric.

## Comparison

The distinction between Linear Regression and Logistic Regression is clear. The former predicts continuous values, such as price, height, or temperature, while the latter predicts a classification, whether binary (for example, expensive versus affordable) or multi class (for example, easy, medium, and hard).

In this project, Linear Regression was used to predict the sale price of houses, achieving an R² of 75% and a MAE of $28,509.31 on a single test split. Logistic Regression was used to classify houses as expensive or affordable, achieving 89.38% accuracy on the same split.

Based on the training and testing done throughout this project, it is evident that a single train test split does not give a complete picture of the dataset and can often be misleading. Both models showed this directly: the linear regression's R² dropped from 75% to 71%, and the logistic regression's accuracy dropped from 89.38% to 86.44%, once evaluated with 10 fold cross validation instead of a single split. In both cases, the single split had made the model look better than it truly is, purely due to how that particular split happened to divide the data. Cross validation mitigates this by averaging results across k folds, giving a more reliable estimate of a model's true performance than any single split can provide, and the fact that both models showed the same pattern suggests this is a general risk of single splits, not something tied to one model or metric.

## Conclusion

Linear Regression and Logistic Regression solve different kinds of problems, continuous prediction versus classification, but both were shown here to be equally vulnerable to misleading results from a single train test split, and both were corrected by the same fix: cross validation.